## Imports

In [181]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

from neuro_fuzzy_toolbox import h_ANFIS, Hybrid_learning_algorithm, SONFIS, EarlyStopping, get_measures, Gaussian_MF

In [182]:
import numpy as np

In [183]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [184]:
from ucimlrepo import fetch_ucirepo

# Heart Disease

## Data

In [185]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

In [186]:
X = X.fillna(value=0)

In [187]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    float64
 12  thal      303 non-null    float64
dtypes: float64(3), int64(10)
memory usage: 30.9 KB


In [188]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [189]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[115  38  25  25   9]


In [190]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[49 17 11 10  4]


In [191]:
scaler = MinMaxScaler(feature_range=(-1, 1))

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [192]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [193]:
loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 16, shuffle = True)
x_train = loader.dataset.tensors[0]
y_train = loader.dataset.tensors[1]
x_train.shape, y_train.shape

(torch.Size([212, 13]), torch.Size([212]))

In [194]:
x_train

tensor([[-0.3750,  1.0000,  1.0000,  ..., -1.0000, -0.3333, -0.1429],
        [ 0.4583, -1.0000,  0.3333,  ..., -1.0000, -1.0000,  1.0000],
        [ 0.0417, -1.0000,  0.3333,  ...,  0.0000, -1.0000, -0.1429],
        ...,
        [ 0.5000, -1.0000,  0.3333,  ..., -1.0000, -1.0000, -0.1429],
        [-0.5000, -1.0000, -0.3333,  ..., -1.0000, -0.3333, -0.1429],
        [ 0.4583,  1.0000,  1.0000,  ...,  0.0000, -0.3333,  1.0000]],
       dtype=torch.float64)

In [195]:
y_train

tensor([1, 0, 0, 0, 1, 1, 2, 0, 0, 0, 0, 0, 1, 2, 2, 0, 2, 0, 0, 1, 0, 1, 0, 0,
        0, 0, 2, 0, 1, 0, 0, 0, 0, 0, 0, 4, 0, 0, 1, 1, 0, 0, 2, 0, 0, 3, 0, 3,
        3, 0, 0, 2, 0, 3, 3, 0, 3, 0, 0, 2, 0, 0, 1, 1, 0, 3, 1, 3, 0, 0, 2, 4,
        0, 1, 0, 0, 1, 1, 4, 0, 3, 0, 3, 2, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0,
        0, 1, 1, 3, 0, 0, 0, 0, 2, 1, 0, 4, 2, 0, 0, 0, 0, 0, 0, 3, 0, 2, 3, 2,
        3, 0, 0, 0, 0, 0, 0, 2, 0, 1, 2, 4, 0, 3, 0, 2, 0, 1, 2, 0, 0, 4, 0, 1,
        0, 0, 0, 3, 1, 1, 0, 0, 1, 3, 0, 0, 0, 2, 0, 0, 1, 3, 0, 2, 2, 1, 0, 3,
        0, 0, 3, 0, 0, 1, 0, 4, 3, 2, 0, 0, 0, 2, 1, 2, 0, 0, 1, 2, 0, 0, 0, 1,
        3, 3, 1, 0, 1, 4, 1, 0, 3, 0, 0, 3, 4, 0, 0, 0, 0, 0, 0, 0])

## Model & Training

### ANFIS

In [196]:
model = h_ANFIS(
    input_size = 13,
    num_mfs = 1,
    outputs = 5,
    rule_reduced = True,
    output_type='multiclass'
)

model.init_premises(x_train)

### Hybrid Learning Algorithm

In [197]:
loss_fn = nn.functional.cross_entropy
epochs = 500
optimizer = torch.optim.AdamW
params = {'lr': 0.0001, 'weight_decay': 0.001}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = EarlyStopping(
    patience=50, 
    delta=0.01
)

In [198]:
trainer = Hybrid_learning_algorithm(
    epochs=epochs,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

### SONFIS

In [199]:
#Ngrow = 30
#dGrow = 0.8
#Nsplit = 30
#eSplit = 0.7
#Nvanish = 8
#lVanish = 3

Ngrow = 8
dGrow = 3.0
Nsplit = 6
eSplit = 2.6
Nvanish = 3
lVanish = 3

max_iterations = 100

anfis_trainer = trainer

validation = 0.2
sonfis_early_stopping = EarlyStopping(patience=30)
last_training_iteration = True

In [200]:
sonfis = SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    validation=validation,
    early_stopping=sonfis_early_stopping,
    last_training_iteration=last_training_iteration
)

In [201]:
%time sonfis(model, loader, verbose=True)

Iteration:   0/100 - loss: 1.977522 - validation loss: 1.639537
 -> ANFIS rules: 1

Iteration:   1/100 - loss: 1.321669 - validation loss: 1.351715
 -> ANFIS rules: 2

Iteration:   2/100 - loss: 1.309066 - validation loss: 1.308539
 -> ANFIS rules: 4

Iteration:   3/100 - loss: 1.466815 - validation loss: 1.487851
 -> ANFIS rules: 8

Iteration:   4/100 - loss: 1.319125 - validation loss: 1.400101
 -> ANFIS rules: 14

Iteration:   5/100 - loss: 1.317898 - validation loss: 1.366237
 -> ANFIS rules: 21

Iteration:   6/100 - loss: 1.291417 - validation loss: 1.335352
 -> ANFIS rules: 28

Iteration:   7/100 - loss: 1.282057 - validation loss: 1.340041
 -> ANFIS rules: 34

Iteration:   8/100 - loss: 1.282903 - validation loss: 1.339394
 -> ANFIS rules: 36

Iteration:   9/100 - loss: 1.282448 - validation loss: 1.339469
 -> ANFIS rules: 40

Iteration:  10/100 - loss: 1.282796 - validation loss: 1.337792
 -> ANFIS rules: 44

Iteration:  11/100 - loss: 1.284069 - validation loss: 1.340539
 -> A

In [202]:
test_measures = get_measures(model, x_test, y_test)

for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.4945054945054945
Precision: 0.4435286935286935
Recall: 0.4945054945054945
F1: 0.46640864376713437
Confusion Matrix: [[38  8  1  2  0]
 [11  4  1  1  0]
 [ 4  2  2  3  0]
 [ 4  3  2  1  0]
 [ 0  0  3  1  0]]


In [203]:
train_measures = get_measures(model, x_train, y_train)

for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.6367924528301887
Precision: 0.6001185585311422
Recall: 0.6367924528301887
F1: 0.6012505945774537
Confusion Matrix: [[105   8   0   2   0]
 [ 19  13   6   0   0]
 [  8   6  11   0   0]
 [  7   2  10   6   0]
 [  1   3   4   1   0]]
